# Cálculo 4 — Sequências, Séries e Equações Diferenciais

Tópicos de **Sequências e Séries** e **Equações Diferenciais Ordinárias**, com foco em visualização de convergência:

1. **Teorema do Confronto** — espremendo uma sequência difícil entre duas sequências fáceis
2. **Séries Convergentes e Divergentes** — somas parciais e testes de convergência (razão, raiz, termo geral)
3. **Sequências e Séries Limitadas** — toda convergente é limitada, mas nem toda limitada converge
4. **Série de Taylor** — aproximação polinomial local de ordem crescente
5. **Série de Fourier** — aproximação por senos e cossenos, com o fenômeno de Gibbs
6. **Fórmula de Parseval** — conservação de energia entre o sinal e seus harmônicos
7. **Equações Diferenciais de 1ª Ordem** — campo de direções e soluções com `sp.dsolve`
8. **Equações Diferenciais de Ordem Superior** — oscilador harmônico e retrato de fase

> **Como usar:** cada seção tem sua própria célula de parâmetros (`# === Parametros ===`); edite e reexecute.

---

## Como usar este notebook

1. **Edite os parâmetros de cada seção:** cada célula de código tem variáveis configuráveis isoladas em um bloco `# === Parametros ===` / `# ==================`.
2. **Reexecute todas as células:** *Kernel → Restart & Run All* (algumas seções reutilizam variáveis de seções anteriores — ex.: a seção de Parseval usa os coeficientes calculados na seção de Fourier).
3. **Sintaxe (SymPy):** use `**` para potência, `sin`, `cos`, `exp`, `log`, `sqrt`, `Abs`, `factorial`. A constante $\pi$ é `pi` e o infinito é `oo`.
4. **Sequências (`n`):** expressões como `1/n`, `(-1)**n`, `sin(n)/n`.
5. **Funções de `x` (Taylor/Fourier):** expressões como `cos(x)`, `exp(-x**2)`, `Abs(x)`.
6. **EDOs:** o lado direito de $y'=f(x,y)$ é escrito em termos de `x` e `y` (ex.: `"x - y"`); o SymPy monta e resolve a equação simbolicamente.

## 0. Configuração

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib import cm
from IPython.display import display

try:
    import plotly.graph_objects as go

    PLOTLY = True
except ImportError:
    PLOTLY = False
    print("plotly nao instalado — execute: pip install plotly")

matplotlib.style.use("ggplot")
%matplotlib inline


def _vals(func_num, pontos):
    '''Avalia uma funcao lambdify sobre um array, garantindo um vetor (broadcast de escalares).'''
    saida = np.asarray(func_num(pontos), dtype=float)
    if saida.shape != pontos.shape:
        saida = np.full(pontos.shape, float(saida))
    return saida


def somas_parciais(a_num, N):
    '''Vetor (ks, S) das somas parciais S_1, ..., S_N de uma sequencia numerica a_num(k).'''
    ks = np.arange(1, N + 1)
    termos = _vals(a_num, ks)
    return ks, np.cumsum(termos)


def cores_gradiente(n_curvas, cmap_nome="viridis", vmin=0.15, vmax=0.92):
    '''Lista de n_curvas cores em gradiente (para sobrepor aproximacoes sucessivas).'''
    cmap = plt.get_cmap(cmap_nome)
    return [cmap(v) for v in np.linspace(vmin, vmax, n_curvas)]


print("Notebook de Calculo 4 configurado: sequencias, series e equacoes diferenciais.")

## 1. Teorema do Confronto (Squeeze Theorem) para Sequências

Se $a_n \le b_n \le c_n$ para todo $n$ suficientemente grande, e

$$\lim_{n\to\infty} a_n = \lim_{n\to\infty} c_n = L$$

então

$$\lim_{n\to\infty} b_n = L$$

também. É a ferramenta clássica para calcular limites de sequências "difíceis" espremendo-as entre duas sequências "fáceis" que convergem para o mesmo valor.

Exemplo clássico: $b_n = \dfrac{\sin n}{n}$ não tem comportamento monótono óbvio, mas fica espremida entre $a_n=-1/n$ e $c_n=1/n$, ambas $\to 0$.

In [ ]:
# === Parametros ===
expr_a_str = "-1/n"
expr_b_str = "sin(n)/n"
expr_c_str = "1/n"
N_max = 60
# ==================

n_sym = sp.symbols("n", positive=True, integer=True)
a_sym = sp.sympify(expr_a_str, locals={"n": n_sym})
b_sym = sp.sympify(expr_b_str, locals={"n": n_sym})
c_sym = sp.sympify(expr_c_str, locals={"n": n_sym})

a_num = sp.lambdify(n_sym, a_sym, "numpy")
b_num = sp.lambdify(n_sym, b_sym, "numpy")
c_num = sp.lambdify(n_sym, c_sym, "numpy")

lim_a = sp.limit(a_sym, n_sym, sp.oo)
lim_b = sp.limit(b_sym, n_sym, sp.oo)
lim_c = sp.limit(c_sym, n_sym, sp.oo)

print("Sequencias do teorema do confronto:")
display(sp.Eq(sp.Symbol("a_n"), a_sym))
display(sp.Eq(sp.Symbol("b_n"), b_sym))
display(sp.Eq(sp.Symbol("c_n"), c_sym))
print(f"\nlim a_n = {lim_a},   lim b_n = {lim_b},   lim c_n = {lim_c}")
if lim_a == lim_c:
    print(f"Como lim a_n = lim c_n = {lim_a}, o Teorema do Confronto garante lim b_n = {lim_a}.")

ns = np.arange(1, N_max + 1)
a_vals = _vals(a_num, ns)
b_vals = _vals(b_num, ns)
c_vals = _vals(c_num, ns)

fig, ax = plt.subplots(figsize=(9, 5), dpi=100, facecolor="w")
ax.plot(ns, a_vals, "o--", color="#d62728", markersize=4, alpha=0.8, label=f"$a_n = {sp.latex(a_sym)}$")
ax.plot(ns, c_vals, "o--", color="#2ca02c", markersize=4, alpha=0.8, label=f"$c_n = {sp.latex(c_sym)}$")
ax.fill_between(ns, a_vals, c_vals, color="gray", alpha=0.15, label="faixa $[a_n, c_n]$")
ax.plot(ns, b_vals, "o-", color="#1f77b4", markersize=5, linewidth=1.6, label=f"$b_n = {sp.latex(b_sym)}$ (espremida)")
ax.axhline(float(lim_a), color="k", linestyle=":", linewidth=1, label=f"L = {float(lim_a):.3g}")
ax.set_xlabel("n")
ax.set_ylabel("valor da sequencia")
ax.set_title("Teorema do Confronto: $a_n \\leq b_n \\leq c_n$", fontsize=12, fontweight="bold")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Séries Convergentes e Divergentes

Dada uma sequência $(a_n)$, a série $\sum a_n$ **converge** se a sequência de somas parciais

$$S_N = \sum_{n=1}^N a_n$$

tem limite finito quando $N\to\infty$. Caso contrário, a série **diverge** (podendo ir a $\pm\infty$ ou oscilar sem se estabilizar).

**Testes de convergência comuns:**

- **Teste da razão (d'Alembert):** $L=\displaystyle\lim_{n\to\infty} \left|\frac{a_{n+1}}{a_n}\right|$; converge absolutamente se $L<1$, diverge se $L>1$ (inconclusivo se $L=1$).
- **Teste da raiz (Cauchy):** $L=\displaystyle\lim_{n\to\infty} \sqrt[n]{|a_n|}$; mesmo critério de decisão.
- **Teste da comparação:** se $0\le a_n \le b_n$ e $\sum b_n$ converge, então $\sum a_n$ converge.
- **Teste do termo geral (condição necessária):** se $\lim a_n \ne 0$, a série diverge.

In [ ]:
# === Parametros ===
N_max = 60
series_exemplo = {
    "geometrica (r=1/2)": "(1/2)**n",
    "harmonica (diverge)": "1/n",
    "alternada  (-1)^n/n": "(-1)**n/n",
}
expr_teste_str = "1/factorial(n)"  # serie usada para ilustrar os testes de convergencia
# ==================

n_sym = sp.symbols("n", positive=True, integer=True)

print("Somas parciais S_N = soma_(k=1 ate N) a_k:\n")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100, facecolor="w")
cores = ["#1f77b4", "#d62728", "#2ca02c"]
for (nome, expr_str), cor in zip(series_exemplo.items(), cores):
    a_sym_i = sp.sympify(expr_str, locals={"n": n_sym})
    a_num_i = sp.lambdify(n_sym, a_sym_i, "numpy")
    ks, S = somas_parciais(a_num_i, N_max)

    try:
        soma_exata = sp.summation(a_sym_i, (n_sym, 1, sp.oo))
    except Exception:
        soma_exata = None
    print(f"  {nome}:  a_n = {a_sym_i},   soma infinita = {soma_exata}")

    axes[0].plot(ks, S, "o-", markersize=3, linewidth=1.3, color=cor, label=nome)
    axes[1].plot(ks, S, "o-", markersize=3, linewidth=1.3, color=cor, label=nome)

axes[0].set_title("Somas parciais $S_N$", fontsize=11, fontweight="bold")
axes[0].set_xlabel("N")
axes[0].set_ylabel("$S_N$")
axes[0].legend(fontsize=8)

axes[1].set_title("Mesmo grafico (escala log em N)", fontsize=11, fontweight="bold")
axes[1].set_xscale("log")
axes[1].set_xlabel("N (escala log)")
axes[1].legend(fontsize=8)

plt.suptitle("Convergencia vs. divergencia de series", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("Testes de convergencia aplicados a:", expr_teste_str)
print("=" * 60)
a_sym_t = sp.sympify(expr_teste_str, locals={"n": n_sym})
display(sp.Eq(sp.Symbol("a_n"), a_sym_t))

razao = sp.simplify(sp.Abs(a_sym_t.subs(n_sym, n_sym + 1)) / sp.Abs(a_sym_t))
try:
    L_razao = sp.limit(razao, n_sym, sp.oo)
except (NotImplementedError, TypeError):
    L_razao = None
print(f"\nTeste da razao: L = lim |a_(n+1)/a_n| = {L_razao}")
try:
    if L_razao is not None and L_razao < 1:
        print("  L < 1  ->  serie converge absolutamente.")
    elif L_razao is not None and L_razao > 1:
        print("  L > 1  ->  serie diverge.")
    elif L_razao is not None:
        print("  L = 1  ->  teste inconclusivo.")
except TypeError:
    pass

raiz = sp.Abs(a_sym_t) ** (sp.Integer(1) / n_sym)
try:
    L_raiz = sp.limit(raiz, n_sym, sp.oo)
except (NotImplementedError, TypeError):
    L_raiz = None
print(f"\nTeste da raiz: L = lim (|a_n|)^(1/n) = {L_raiz}")

try:
    lim_termo = sp.limit(a_sym_t, n_sym, sp.oo)
except (NotImplementedError, TypeError):
    lim_termo = None
print(f"\nTeste do termo geral: lim a_n = {lim_termo}")
if lim_termo is not None and lim_termo != 0:
    print("  lim a_n != 0  ->  serie diverge (condicao necessaria falha).")

## 3. Sequências (e Séries) Limitadas

Uma sequência $(a_n)$ é **limitada** se existe $M>0$ tal que $|a_n|\le M$ para todo $n$. Todo **limite convergente é limitado**, mas a recíproca é falsa: existem sequências limitadas que não convergem (oscilam para sempre). O exemplo canônico é $a_n=(-1)^n$, que fica presa entre $-1$ e $1$ mas nunca converge.

Para séries, as **somas parciais limitadas** ($|S_N|\le M$ para todo $N$) são uma condição necessária, mas não suficiente, para a convergência — é também um ingrediente do teste da série alternada de Leibniz.

In [ ]:
# === Parametros ===
N_max = 40
exemplos = {
    "Convergente:  a_n = 1/n": ("1/n", 1.0),
    "Limitada, NAO convergente:  a_n = (-1)^n": ("(-1)**n", 1.0),
    "Ilimitada:  a_n = sqrt(n)": ("sqrt(n)", None),
}
# ==================

n_sym = sp.symbols("n", positive=True, integer=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=100, facecolor="w")
for ax, (nome, (expr_str, M)) in zip(axes, exemplos.items()):
    a_sym_i = sp.sympify(expr_str, locals={"n": n_sym})
    a_num_i = sp.lambdify(n_sym, a_sym_i, "numpy")
    ns = np.arange(1, N_max + 1)
    vals = _vals(a_num_i, ns)

    ax.plot(ns, vals, "o-", color="#1f77b4", markersize=4, linewidth=1.2)
    if M is not None:
        ax.axhline(M, color="red", linestyle="--", linewidth=1, label=f"M = {M:g}")
        ax.axhline(-M, color="red", linestyle="--", linewidth=1)
        ax.fill_between(ns, -M, M, color="red", alpha=0.07)
        ax.legend(fontsize=8)
    ax.set_title(nome, fontsize=9.5, fontweight="bold")
    ax.set_xlabel("n")
    ax.set_ylabel("a_n")

plt.suptitle("Convergente vs. limitada (sem convergir) vs. ilimitada", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Toda sequencia convergente e limitada, mas nem toda sequencia limitada converge:")
print("  (-1)^n oscila entre -1 e 1 para sempre      -> limitada, NAO convergente.")
print("  sqrt(n) cresce sem cota quando n -> infinito -> ilimitada, NAO convergente.")
print("  1/n -> 0                                     -> convergente (logo, limitada).")

## 4. Expansão em Séries de Potências — Série de Taylor

A **série de Taylor** de $f$ em torno de $x=a$ é

$$f(x) \approx T_N(x) = \sum_{n=0}^{N} \frac{f^{(n)}(a)}{n!}(x-a)^n$$

Quanto maior $N$, melhor a aproximação local de $f$ (dentro do raio de convergência). Abaixo, $T_N(x)$ é construído simbolicamente pelo SymPy (`sp.series`) para ordens crescentes e comparado a $f(x)$.

In [ ]:
# === Parametros ===
expr_str = "cos(x)"
a_taylor = 0
ordens = [1, 3, 5, 10]
raio_plot = 6.0
# ==================

x_sym = sp.symbols("x")
f_sym = sp.sympify(expr_str)
f_num = sp.lambdify(x_sym, f_sym, "numpy")

print(f"f(x) = {expr_str},  expandida em torno de a = {a_taylor}\n")
polinomios = []
for N in ordens:
    poli = sp.series(f_sym, x_sym, a_taylor, N + 1).removeO()
    polinomios.append(poli)
    print(f"T_{N}(x) =")
    display(sp.Eq(sp.Symbol(f"T_{N}"), poli))

xs = np.linspace(a_taylor - raio_plot, a_taylor + raio_plot, 400)
f_vals = _vals(f_num, xs)
cores = cores_gradiente(len(ordens), "plasma")

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=100, facecolor="w")
for ax, N, poli, cor in zip(axes.flat, ordens, polinomios, cores):
    poli_num = sp.lambdify(x_sym, poli, "numpy")
    T_vals = _vals(poli_num, xs)
    ax.plot(xs, f_vals, color="black", linewidth=2, label="f(x)")
    ax.plot(xs, T_vals, color=cor, linewidth=2, linestyle="--", label=f"$T_{{{N}}}(x)$")
    ax.axvline(a_taylor, color="gray", linestyle=":", linewidth=1)
    ax.set_ylim(np.nanmin(f_vals) - 2, np.nanmax(f_vals) + 2)
    ax.set_title(f"Ordem N = {N}", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_xlabel("x")

plt.suptitle(f"Aproximacoes de Taylor de f(x) = {expr_str} em a = {a_taylor}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100, facecolor="w")
ax.plot(xs, f_vals, color="black", linewidth=2.5, label="f(x)", zorder=5)
for N, poli, cor in zip(ordens, polinomios, cores):
    poli_num = sp.lambdify(x_sym, poli, "numpy")
    T_vals = _vals(poli_num, xs)
    ax.plot(xs, T_vals, color=cor, linewidth=1.6, label=f"$T_{{{N}}}(x)$")
ax.axvline(a_taylor, color="gray", linestyle=":", linewidth=1)
ax.set_ylim(np.nanmin(f_vals) - 2, np.nanmax(f_vals) + 2)
ax.set_title("Convergencia progressiva de T_N(x) para f(x)", fontsize=12, fontweight="bold")
ax.set_xlabel("x")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

## 5. Expansão em Séries de Potências — Série de Fourier

Para uma função periódica de período $T=2L$, a série de Fourier é

$$f(x) \approx \frac{a_0}{2} + \sum_{n=1}^{N}\left[a_n\cos\!\left(\frac{n\pi x}{L}\right) + b_n \sin\!\left(\frac{n\pi x}{L}\right)\right]$$

com coeficientes

$$a_0=\frac{1}{L}\int_{-L}^{L} f(x)\,dx,\qquad a_n=\frac{1}{L}\int_{-L}^{L} f(x)\cos\!\left(\frac{n\pi x}{L}\right)dx,\qquad b_n=\frac{1}{L}\int_{-L}^{L} f(x)\sin\!\left(\frac{n\pi x}{L}\right)dx$$

Abaixo os coeficientes são calculados simbolicamente com `sp.integrate` para uma onda periódica configurável (quadrada, dente-de-serra ou triangular), e a soma parcial $S_N$ é comparada à função original para $N$ crescente.

In [ ]:
# === Parametros ===
tipo_onda = "quadrada"  # opcoes: "quadrada", "dente_de_serra", "triangular"
L_fourier_str = "pi"  # meio-periodo L (T = 2L); aceita expressoes simbolicas, ex.: "pi", "2", "3*pi/2"
ordens_fourier = [1, 3, 9, 30]
# ==================

x_sym = sp.symbols("x", real=True)
n_sym = sp.symbols("n", positive=True, integer=True)
Lf = sp.sympify(L_fourier_str)

if tipo_onda == "quadrada":
    f_fourier = sp.Piecewise((-1, x_sym < 0), (1, True))
elif tipo_onda == "dente_de_serra":
    f_fourier = x_sym
elif tipo_onda == "triangular":
    f_fourier = sp.Abs(x_sym)
else:
    raise ValueError("tipo_onda deve ser 'quadrada', 'dente_de_serra' ou 'triangular'")

a0_sym = (1 / Lf) * sp.integrate(f_fourier, (x_sym, -Lf, Lf))
an_sym = sp.simplify((1 / Lf) * sp.integrate(f_fourier * sp.cos(n_sym * sp.pi * x_sym / Lf), (x_sym, -Lf, Lf)))
bn_sym = sp.simplify((1 / Lf) * sp.integrate(f_fourier * sp.sin(n_sym * sp.pi * x_sym / Lf), (x_sym, -Lf, Lf)))

print(f"Onda escolhida: {tipo_onda},  periodo T = 2L = {2 * Lf}")
display(sp.Eq(sp.Symbol("a_0"), a0_sym))
display(sp.Eq(sp.Symbol("a_n"), an_sym))
display(sp.Eq(sp.Symbol("b_n"), bn_sym))

f_num = sp.lambdify(x_sym, f_fourier, "numpy")
an_num = sp.lambdify(n_sym, an_sym, "numpy")
bn_num = sp.lambdify(n_sym, bn_sym, "numpy")
Lf_val = float(Lf)
a0_val = float(a0_sym)


def serie_parcial(xs, N):
    '''Soma parcial S_N(x) da serie de Fourier nos pontos xs.'''
    S = np.full_like(xs, a0_val / 2.0, dtype=float)
    for k in range(1, N + 1):
        ak = float(an_num(k))
        bk = float(bn_num(k))
        S = S + ak * np.cos(k * np.pi * xs / Lf_val) + bk * np.sin(k * np.pi * xs / Lf_val)
    return S


xs = np.linspace(-2 * Lf_val, 2 * Lf_val, 800)
xs_periodica = ((xs + Lf_val) % (2 * Lf_val)) - Lf_val
f_vals = _vals(f_num, xs_periodica)
cores = cores_gradiente(len(ordens_fourier), "viridis")

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=100, facecolor="w")
for ax, N, cor in zip(axes.flat, ordens_fourier, cores):
    S_vals = serie_parcial(xs, N)
    ax.plot(xs, f_vals, color="black", linewidth=1.8, label="f(x) (periodica)")
    ax.plot(xs, S_vals, color=cor, linewidth=1.6, label=f"$S_{{{N}}}(x)$")
    ax.set_title(f"N = {N} harmonicos", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_xlabel("x")

plt.suptitle(f"Serie de Fourier da onda {tipo_onda}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Fenômeno de Gibbs:** perto de uma descontinuidade de salto, a soma parcial de Fourier sempre ultrapassa o valor do salto em cerca de **9%**, não importa quantos harmônicos sejam somados — o *overshoot* não desaparece, apenas se estreita em torno do salto.

In [ ]:
# === Parametros (fenomeno de Gibbs, mais visivel na onda quadrada) ===
ordens_gibbs = [5, 15, 45, 135]
zoom = 0.6  # largura da janela de zoom em torno do salto x=0, em unidades de L
# ==================

if tipo_onda == "quadrada":
    xs_zoom = np.linspace(-zoom * Lf_val, zoom * Lf_val, 2000)
    f_zoom = _vals(f_num, ((xs_zoom + Lf_val) % (2 * Lf_val)) - Lf_val)
    cores_g = cores_gradiente(len(ordens_gibbs), "inferno")

    overshoot_teorico = 1.0 + 0.0895  # ~8.95% acima do salto (constante de Gibbs)

    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100, facecolor="w")
    ax.plot(xs_zoom, f_zoom, color="black", linewidth=2, label="f(x)", zorder=5)
    for N, cor in zip(ordens_gibbs, cores_g):
        S_zoom = serie_parcial(xs_zoom, N)
        ax.plot(xs_zoom, S_zoom, color=cor, linewidth=1.4, label=f"$S_{{{N}}}(x)$")

    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1)
    ax.axhline(overshoot_teorico, color="red", linestyle="--", linewidth=1, label=f"overshoot teorico ~ {overshoot_teorico:.3f}")
    ax.set_title("Fenomeno de Gibbs perto da descontinuidade (x=0)", fontsize=12, fontweight="bold")
    ax.set_xlabel("x")
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()

    print("Overshoot teorico do fenomeno de Gibbs: ~8.95% do tamanho do salto,")
    print("e esse valor NAO diminui quando N aumenta -- apenas a regiao do overshoot encolhe.")
else:
    print("O fenomeno de Gibbs e mais evidente perto de uma descontinuidade de salto.")
    print("Troque tipo_onda = 'quadrada' na celula de configuracao da serie de Fourier para visualiza-lo.")

## 6. Fórmula de Parseval

A identidade de Parseval relaciona a energia média de um sinal periódico com a energia de seus harmônicos de Fourier:

$$\frac{1}{T}\int_{0}^{T} |f(x)|^2\,dx \;=\; \frac{a_0^2}{4} + \frac{1}{2}\sum_{n=1}^{\infty}\left(a_n^2+b_n^2\right)$$

Ou seja: a energia total do sinal é igual à soma das energias de cada harmônico — nenhuma energia "se perde" na decomposição em senos e cossenos. Abaixo comparamos a energia exata (integral de $f^2$) com a soma truncada dos $N$ primeiros harmônicos calculados na seção anterior.

In [ ]:
# === Parametros ===
N_parseval = 60
# ==================

energia_exata = sp.nsimplify(1 / (2 * Lf) * sp.integrate(f_fourier**2, (x_sym, -Lf, Lf)))
energia_exata_val = float(energia_exata)
print(f"Energia exata (1/T) * integral de f(x)^2 sobre um periodo = {energia_exata}  ({energia_exata_val:.6f})")

energia_acumulada = np.zeros(N_parseval)
acumulado = a0_val**2 / 4.0
for k in range(1, N_parseval + 1):
    ak = float(an_num(k))
    bk = float(bn_num(k))
    acumulado += 0.5 * (ak**2 + bk**2)
    energia_acumulada[k - 1] = acumulado

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100, facecolor="w")
ax.plot(np.arange(1, N_parseval + 1), energia_acumulada, "o-", color="#1f77b4", markersize=4, label="energia acumulada ate N harmonicos")
ax.axhline(energia_exata_val, color="red", linestyle="--", linewidth=1.5, label=f"energia exata = {energia_exata_val:.4f}")
ax.set_xlabel("N (numero de harmonicos somados)")
ax.set_ylabel("energia")
ax.set_title(f"Convergencia de Parseval — onda {tipo_onda}", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

erro_final = abs(energia_acumulada[-1] - energia_exata_val)
print(f"\nErro apos N={N_parseval} harmonicos: {erro_final:.3e}")
print("A soma truncada converge para a energia exata conforme N cresce, confirmando a formula de Parseval.")

## 7. Equações Diferenciais de 1ª Ordem

Uma EDO de 1ª ordem $y' = f(x,y)$ define, em cada ponto do plano, uma inclinação — o **campo de direções** (*slope field*). Resolvendo simbolicamente com `sp.dsolve`, obtemos a família de soluções $y(x)$; cada condição inicial $y(x_0)=y_0$ seleciona uma curva específica dentro dessa família, que deve ser tangente ao campo de direções em todo ponto.

In [ ]:
# === Parametros ===
rhs_str = "x - y"  # dy/dx = f(x, y) -- escreva em termos de 'x' e 'y'
x0_ci = 0.0
condicoes_iniciais = [-2.0, -1.0, 0.0, 1.0, 2.0]  # valores de y(x0_ci)
XMIN, XMAX, YMIN, YMAX = -3.0, 3.0, -3.0, 3.0
# ==================

x_sym, y_sym = sp.symbols("x y", real=True)
rhs_xy = sp.sympify(rhs_str, locals={"x": x_sym, "y": y_sym})
f_slope = sp.lambdify((x_sym, y_sym), rhs_xy, "numpy")

y_f = sp.Function("y")
ode_eq = sp.Eq(y_f(x_sym).diff(x_sym), rhs_xy.subs(y_sym, y_f(x_sym)))
sol_geral = sp.dsolve(ode_eq, y_f(x_sym))

print("EDO de 1a ordem:")
display(ode_eq)
print("Solucao geral:")
display(sol_geral)

C1 = sp.Symbol("C1")
curvas = []
for y0 in condicoes_iniciais:
    eq_c1 = sp.Eq(sol_geral.rhs.subs(x_sym, x0_ci), y0)
    sols_c1 = sp.solve(eq_c1, C1)
    if not sols_c1:
        continue
    y_particular = sol_geral.rhs.subs(C1, sols_c1[0])
    curvas.append((y0, sp.lambdify(x_sym, y_particular, "numpy")))

Xg, Yg = np.meshgrid(np.linspace(XMIN, XMAX, 20), np.linspace(YMIN, YMAX, 20))
with np.errstate(divide="ignore", invalid="ignore"):
    Dy = np.asarray(f_slope(Xg, Yg), dtype=float)
Dx = np.ones_like(Dy)
Dy[~np.isfinite(Dy)] = 0.0
norma = np.sqrt(Dx**2 + Dy**2)
norma[norma < 1e-12] = 1.0
Dxn, Dyn = Dx / norma, Dy / norma

fig, ax = plt.subplots(figsize=(8.5, 7), dpi=100, facecolor="w")
ax.quiver(Xg, Yg, Dxn, Dyn, angles="xy", color="gray", alpha=0.6, width=0.0035, scale=28)
cores_c = cores_gradiente(max(len(curvas), 1), "cool")
xs_plot = np.linspace(XMIN, XMAX, 300)
for (y0, y_num), cor in zip(curvas, cores_c):
    ax.plot(xs_plot, _vals(y_num, xs_plot), color=cor, linewidth=2, label=f"y({x0_ci:g}) = {y0:g}")
ax.set_xlim(XMIN, XMAX)
ax.set_ylim(YMIN, YMAX)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Campo de direcoes e solucoes de y' = {rhs_str}", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

## 8. Equações Diferenciais de Ordem Superior

O **oscilador harmônico amortecido** é o exemplo clássico de EDO linear de 2ª ordem:

$$y'' + 2\zeta\omega_0\, y' + \omega_0^2\, y = 0$$

onde $\omega_0$ é a frequência natural e $\zeta$ é o fator de amortecimento. Dependendo de $\zeta$, o sistema é:

| $\zeta$ | Regime |
|---|---|
| $\zeta = 0$ | não amortecido (oscila para sempre) |
| $0<\zeta<1$ | subamortecido (oscila com amplitude decrescente) |
| $\zeta = 1$ | criticamente amortecido |
| $\zeta > 1$ | superamortecido |

Resolvemos com `sp.dsolve` e condições iniciais $y(0)$, $y'(0)$, visualizando $y(t)$ e o **retrato de fase** $(y,\dot y)$.

In [ ]:
# === Parametros ===
omega0 = 2.0
zeta = 0.15
y0_ci, v0_ci = 1.0, 0.0
t_max = 20.0
# ==================

t_sym = sp.symbols("t", real=True)
y_f2 = sp.Function("y")

# nsimplify evita problemas numericos do dsolve com coeficientes em ponto flutuante
zeta_r = sp.nsimplify(zeta, rational=True)
omega0_r = sp.nsimplify(omega0, rational=True)

eq_osc = sp.Eq(
    y_f2(t_sym).diff(t_sym, 2) + 2 * zeta_r * omega0_r * y_f2(t_sym).diff(t_sym) + omega0_r**2 * y_f2(t_sym),
    0,
)
sol_osc = sp.dsolve(
    eq_osc,
    y_f2(t_sym),
    ics={
        y_f2(0): sp.nsimplify(y0_ci, rational=True),
        y_f2(t_sym).diff(t_sym).subs(t_sym, 0): sp.nsimplify(v0_ci, rational=True),
    },
)

if zeta == 0:
    regime = "nao amortecido"
elif zeta < 1:
    regime = "subamortecido"
elif zeta == 1:
    regime = "criticamente amortecido"
else:
    regime = "superamortecido"

print(f"Oscilador harmonico ({regime}):  omega0 = {omega0},  zeta = {zeta}")
display(eq_osc)
print("Solucao:")
display(sol_osc)

y_num2 = sp.lambdify(t_sym, sol_osc.rhs, "numpy")
dy_sym = sp.diff(sol_osc.rhs, t_sym)
dy_num2 = sp.lambdify(t_sym, dy_sym, "numpy")

ts = np.linspace(0, t_max, 1000)
y_ts = _vals(y_num2, ts)
dy_ts = _vals(dy_num2, ts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=100, facecolor="w")
axes[0].plot(ts, y_ts, color="#1f77b4", linewidth=2)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_xlabel("t")
axes[0].set_ylabel("y(t)")
axes[0].set_title(f"Resposta no tempo ({regime})", fontsize=11, fontweight="bold")

axes[1].plot(y_ts, dy_ts, color="#d62728", linewidth=1.8)
axes[1].plot([y_ts[0]], [dy_ts[0]], "o", color="green", markersize=8, label="inicio")
axes[1].plot([0], [0], "k*", markersize=12, label="equilibrio")
axes[1].set_xlabel("y")
axes[1].set_ylabel("dy/dt")
axes[1].set_title("Retrato de fase", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].set_aspect("equal", adjustable="datalim")

plt.suptitle("Oscilador harmonico de 2a ordem", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()